# Brazilian Marketplace (Olist) — Business Analysis

Olist is a Brazilian marketplace connecting small businesses to a wide network of online sales channels. This notebook analyzes ~100,000 orders placed on the Olist Store between 2016 and 2018, using the public [Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0).

**Business question:** How can Olist optimize its operational and marketing strategy to improve profitability, increase customer satisfaction, and reduce delivery times, based on historical sales data, customer reviews, and logistics metrics?

**Method:** SQL on Databricks — data modeling, cleaning, revenue analysis, AOV, CLV, RFM segmentation, churn rate, and correlation studies.

## 1. Setup & Schema Overview

The 9 source CSVs were loaded as tables in `workspace.brazilian_marketplace`. Before analyzing, let's confirm the schema of each table matches the dataset documentation.

In [0]:
-- I'm checking the schema of every table I loaded, to make sure it matches what I expect from the dataset documentation before I do anything else with it

SELECT table_name, column_name, data_type, ordinal_position
FROM workspace.information_schema.columns
WHERE table_schema = 'brazilian_marketplace'
ORDER BY table_name, ordinal_position;

The schema matches the dataset documentation for 7 of 9 tables. Two exceptions worth noting:

- `product_category_name_translation` carries an extra `dummy_column` (a constant value on every row, not useful for anything). I'll exclude it from all queries going forward.
- `reviews` only has `review_id`, `order_id`, `review_score` — the source file for reviews is a lighter version that skips the free-text comments and survey dates. That's fine here, since this analysis only needs the review score.

Row counts per table, as a sanity check against the documented dataset size:


In [0]:
-- Quick sanity check: do my row counts match what the Olist dataset documentation says each table should have?

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM workspace.brazilian_marketplace.customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM workspace.brazilian_marketplace.geolocation
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.brazilian_marketplace.order_items
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.brazilian_marketplace.orders
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.brazilian_marketplace.payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.brazilian_marketplace.reviews
UNION ALL SELECT 'products', COUNT(*) FROM workspace.brazilian_marketplace.products
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.brazilian_marketplace.sellers
UNION ALL SELECT 'product_category_name_translation', COUNT(*) FROM workspace.brazilian_marketplace.product_category_name_translation;

All counts match the documented Olist dataset sizes (e.g. 99,441 customers and orders, 112,650 order items, 1,000,163 geolocation records) — the load is clean.

## 2. Primary Key Identification

For each table, a column is a primary key candidate if `COUNT(*) = COUNT(DISTINCT column)` — every value is present and unique.

In [0]:
-- For each table, I'm testing whether a column could be a primary key: if COUNT(*) equals COUNT(DISTINCT column), every value is unique and present, so it's a valid single-column key

SELECT 'customers' AS table_name, 'customer_id' AS candidate_column,
       COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_non_null,
       COUNT(*) = COUNT(DISTINCT customer_id) AS is_primary_key
FROM workspace.brazilian_marketplace.customers
UNION ALL
SELECT 'orders', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.orders
UNION ALL
SELECT 'products', 'product_id', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) = COUNT(DISTINCT product_id)
FROM workspace.brazilian_marketplace.products
UNION ALL
SELECT 'sellers', 'seller_id', COUNT(*), COUNT(DISTINCT seller_id), COUNT(*) = COUNT(DISTINCT seller_id)
FROM workspace.brazilian_marketplace.sellers
UNION ALL
SELECT 'reviews', 'review_id', COUNT(*), COUNT(DISTINCT review_id), COUNT(*) = COUNT(DISTINCT review_id)
FROM workspace.brazilian_marketplace.reviews
UNION ALL
SELECT 'product_category_name_translation', 'product_category_name', COUNT(*), COUNT(DISTINCT product_category_name), COUNT(*) = COUNT(DISTINCT product_category_name)
FROM workspace.brazilian_marketplace.product_category_name_translation
UNION ALL
SELECT 'order_items', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'geolocation', 'geolocation_zip_code_prefix', COUNT(*), COUNT(DISTINCT geolocation_zip_code_prefix), COUNT(*) = COUNT(DISTINCT geolocation_zip_code_prefix)
FROM workspace.brazilian_marketplace.geolocation;

Four tables came back `false` above. For `order_items`, `payments`, and `geolocation` this is expected (an order can have several items or payment installments; a zip prefix covers many lat/lng points). But `reviews.review_id` repeating is unexpected — a review should be unique. Let's look closer before settling on a composite key.


In [0]:
-- I want to see which review_ids repeat and how often, to understand why review_id alone isn't unique
SELECT review_id, COUNT(*) AS n_occurrences
FROM workspace.brazilian_marketplace.reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY n_occurrences DESC
LIMIT 10;

In [0]:
-- Let's look at the actual rows behind one of these duplicated review_ids, to see what's different between the copies
SELECT *
FROM workspace.brazilian_marketplace.reviews
WHERE review_id IN (
  SELECT review_id
  FROM workspace.brazilian_marketplace.reviews
  GROUP BY review_id
  HAVING COUNT(*) > 1
)
ORDER BY review_id
LIMIT 20;

The duplicates differ by `order_id`: Olist occasionally ties one review survey to more than one order (e.g. orders placed close together get bundled into a single satisfaction survey). So `review_id` alone isn't a key, but `review_id` + `order_id` together should be. Let's confirm that, along with the other composite keys.


In [0]:
-- Checking composite keys for the three tables that failed the single-column test: order_id + order_item_id for order_items, order_id + payment_sequential for payments, review_id + order_id for reviews
SELECT 'order_items' AS table_name, COUNT(*) AS total_rows,
       COUNT(DISTINCT CONCAT(order_id, '-', order_item_id)) AS distinct_composite
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', COUNT(*), COUNT(DISTINCT CONCAT(order_id, '-', payment_sequential))
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT CONCAT(review_id, '-', order_id))
FROM workspace.brazilian_marketplace.reviews;


## Primary Key Summary

| Table | Primary key |
|---|---|
| customers | `customer_id` |
| orders | `order_id` |
| products | `product_id` |
| sellers | `seller_id` |
| product_category_name_translation | `product_category_name` |
| order_items | `order_id` + `order_item_id` (composite) |
| payments | `order_id` + `payment_sequential` (composite) |
| reviews | `review_id` + `order_id` (composite) |
| geolocation | no single natural key — it's a reference lookup table (many lat/lng points share a zip prefix); not decomposed further, since it's only ever joined on `geolocation_zip_code_prefix` |

## 3. ER Schema / Data Model

Relationships between the tables:

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : places
    ORDERS ||--o{ ORDER_ITEMS : contains
    ORDERS ||--o{ PAYMENTS : "paid via"
    ORDERS ||--o{ REVIEWS : receives
    ORDER_ITEMS }o--|| PRODUCTS : references
    ORDER_ITEMS }o--|| SELLERS : "sold by"
    PRODUCTS }o--|| PRODUCT_CATEGORY_NAME_TRANSLATION : "category in"
    CUSTOMERS }o--|| GEOLOCATION : "zip code in"
    SELLERS }o--|| GEOLOCATION : "zip code in"
```

## 4. Data Cleaning — Missing Values

I'll check every table for NULLs across all of its columns in one pass, then decide table by table whether a NULL can actually be fixed or has to stay as-is.

In [0]:
-- I'm checking, for every table, how many rows have at least one NULL in any column - a quick way to see where missing data actually is before deciding what to do about it
SELECT 'customers' AS table_name, COUNT(*) AS rows_with_nulls
FROM workspace.brazilian_marketplace.customers
WHERE customer_id IS NULL OR customer_unique_id IS NULL OR customer_zip_code_prefix IS NULL OR customer_city IS NULL OR customer_state IS NULL
UNION ALL
SELECT 'geolocation', COUNT(*)
FROM workspace.brazilian_marketplace.geolocation
WHERE geolocation_zip_code_prefix IS NULL OR geolocation_lat IS NULL OR geolocation_lng IS NULL OR geolocation_city IS NULL OR geolocation_state IS NULL
UNION ALL
SELECT 'order_items', COUNT(*)
FROM workspace.brazilian_marketplace.order_items
WHERE order_id IS NULL OR order_item_id IS NULL OR product_id IS NULL OR seller_id IS NULL OR shipping_limit_date IS NULL OR price IS NULL OR freight_value IS NULL
UNION ALL
SELECT 'orders', COUNT(*)
FROM workspace.brazilian_marketplace.orders
WHERE order_id IS NULL OR customer_id IS NULL OR order_status IS NULL OR order_purchase_timestamp IS NULL OR order_approved_at IS NULL OR order_delivered_carrier_date IS NULL OR order_delivered_customer_date IS NULL OR order_estimated_delivery_date IS NULL
UNION ALL
SELECT 'payments', COUNT(*)
FROM workspace.brazilian_marketplace.payments
WHERE order_id IS NULL OR payment_sequential IS NULL OR payment_type IS NULL OR payment_installments IS NULL OR payment_value IS NULL
UNION ALL
SELECT 'reviews', COUNT(*)
FROM workspace.brazilian_marketplace.reviews
WHERE review_id IS NULL OR order_id IS NULL OR review_score IS NULL
UNION ALL
SELECT 'products', COUNT(*)
FROM workspace.brazilian_marketplace.products
WHERE product_id IS NULL OR product_category_name IS NULL OR product_name_lenght IS NULL OR product_description_lenght IS NULL OR product_photos_qty IS NULL OR product_weight_g IS NULL OR product_length_cm IS NULL OR product_height_cm IS NULL OR product_width_cm IS NULL
UNION ALL
SELECT 'sellers', COUNT(*)
FROM workspace.brazilian_marketplace.sellers
WHERE seller_id IS NULL OR seller_zip_code_prefix IS NULL OR seller_city IS NULL OR seller_state IS NULL
UNION ALL
SELECT 'product_category_name_translation', COUNT(*)
FROM workspace.brazilian_marketplace.product_category_name_translation
WHERE product_category_name IS NULL OR product_category_name_english IS NULL;

`orders` and `products` are the tables most likely to show NULLs here: some orders were never approved or delivered, so their date columns are legitimately empty, and a small number of products are missing their category name and dimensions in the source data. Without more information from Olist, most of these can't be fixed and are left as-is — that's a real, documented limitation of the dataset, not something to paper over.

The one gap worth closing is the missing **English category translations** in `products`.

Products only carry the Portuguese category name. I'm adding an English version by joining in the translation table — and since a couple of categories don't have a translation on record, I'm filling those in by hand, with anything still unmatched labeled `'N/A'` so it doesn't silently disappear from later analysis.


In [0]:
-- Rebuilding products with an English category column: joined from the translation table, with the two known missing translations filled in by hand, and 'N/A' for anything left unmatched. Safe to run this as many times as I want, from any starting point.
CREATE OR REPLACE TABLE workspace.brazilian_marketplace.products AS
SELECT
  p.product_id,
  p.product_category_name,
  p.product_name_lenght,
  p.product_description_lenght,
  p.product_photos_qty,
  p.product_weight_g,
  p.product_length_cm,
  p.product_height_cm,
  p.product_width_cm,
  COALESCE(
    t.product_category_name_english,
    CASE
      WHEN p.product_category_name = 'portateis_cozinha_e_preparadores_de_alimentos' THEN 'kitchen_and_food_preparation_portable_devices'
      WHEN p.product_category_name = 'pc_gamer' THEN 'gaming_pc'
      ELSE 'N/A'
    END
  ) AS product_category_name_eng
FROM workspace.brazilian_marketplace.products AS p
LEFT JOIN workspace.brazilian_marketplace.product_category_name_translation AS t
  ON p.product_category_name = t.product_category_name;

In [0]:
-- Quick check that every product now has an English category: this should return 0
SELECT COUNT(*) AS still_missing
FROM workspace.brazilian_marketplace.products
WHERE product_category_name_eng IS NULL;

## 5. Data Cleaning — Duplicates

I already answered this back in Step 2: every table with a natural identifier (`customer_id`, `order_id`, `product_id`, `seller_id`, `review_id` + `order_id`, etc.) came back unique when tested as a primary key candidate. So there's no separate duplicate-row problem here — the primary key check *is* the duplicate check for this dataset.

## 6. Revenue Analysis — Total Revenue, Valid Order Statuses, Time Span

Before calculating revenue, I need to know which order statuses actually represent completed sales, and how much history the data covers.

In [0]:
-- Let's see what order statuses exist, so I know which ones should count toward revenue
SELECT DISTINCT order_status
FROM workspace.brazilian_marketplace.orders
ORDER BY order_status;

Olist's own documentation treats only `"delivered"` orders as completed sales — everything else (created, shipped, canceled, etc.) hasn't actually generated revenue yet, or never will. I'll filter on that status for every revenue calculation from here on.


In [0]:
-- Total revenue: summing payment_value for delivered orders only
SELECT ROUND(SUM(p.payment_value), 0) AS total_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';

That revenue figure only means something with a time frame attached. Let's find the first and last purchase dates, and the span between them in days, weeks, months, and years.


In [0]:
-- Using datediff() with a unit argument to get the gap between the first and last purchase in several different units at once
SELECT
  MIN(order_purchase_timestamp) AS started_time,
  MAX(order_purchase_timestamp) AS ended_time,
  DATEDIFF(DAY, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS days,
  DATEDIFF(WEEK, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS weeks,
  DATEDIFF(MONTH, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS months,
  DATEDIFF(YEAR, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS years
FROM workspace.brazilian_marketplace.orders;

This gives me the full picture: how much revenue Olist generated, and over what period. That's the baseline everything else in this analysis builds on.
